## The thread behind the order: cells 1 to 5 were about trusting the data (is it complete, are there duplicates, how big is it, how do we split without leaking). Cells 6 to 7 were about the audio format. Cells 8 to 11 were about the text labels, and they produced the cleaning rules. Cell 13 applied all of it. This is also your Q1a answer (“what you did to process the data and why”)

isbuplicate - The old download script names files by user_id. If a user has two recordings, the second one is skipped, so we had to know whether any were lost.
104 rows, 102 unique users, 2 recordings lost

In [ ]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")

print("rows:", len(df))
print("unique user_id:", df.user_id.nunique())
print("unique recording_id:", df.recording_id.nunique())

dup = df[df.duplicated("user_id", keep=False)].sort_values("user_id")
print("rows sharing a user_id:", len(dup))
print(dup[["user_id", "recording_id", "duration"]].head(20))

# files lost by the old naming (<user_id>_audio.wav)
print("recordings lost to collisions:", len(df) - df.user_id.nunique())

duration- Josh says ~10 h, but your sheet summed to ~21.9 h. We had to check for duplicate rows before trusting either number.
No duplicates, 21.89 h by the CSV

In [ ]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")
print("rows:", len(df))
print("unique recording_id:", df.recording_id.nunique())
print("unique user_id:", df.user_id.nunique())
print("duplicate rows:", df.duplicated().sum())
print("total hours:", df.duration.sum() / 3600)
print("rows with missing duration:", df.duration.isna().sum())

pairing- The rows came in pairs with the same duration. If they’re the same conversation, a split by row would leak the same conversation into train and validation.
All 52 folders have exactly 2 recordings

In [ ]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")
df["folder"] = df.rec_url_gcp.str.extract(r"upload_goai/(\d+)/")[0]
g = df.groupby("folder").agg(n=("recording_id", "size"),
                             users=("user_id", list),
                             durs=("duration", list))
print(g.n.value_counts())
print(g[g.n != 2])

Union-find groups- Speakers 93626 and 147403 have recordings in two folders, so splitting by folder alone would still leak their voices. Groups fix that.
50 groups

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for _, r in df.iterrows():
    parent[find(f"f{r.folder}")] = find(f"u{r.user_id}")

df["group"] = [find(f"f{f}") for f in df.folder]
print("groups:", df.group.nunique())   # expect 50 (52 folders, 2 merges)
# df['group'].head()

Transcribed hours-Training only uses audio that has a label, so the real dataset size is the labelled time, not the file length.	
(12.11 h): for every segment in the transcripts, I took end - start and added them all up. That’s the amount of audio that has a text label.
12.11 h over 102 files	

In [ ]:
import json, glob
tot, n = 0, 0
for f in glob.glob("../data/transcripts/*.json"):
    n += 1
    for s in json.load(open(f, encoding="utf-8")):
        tot += s["end"] - s["start"]
print("files:", n, "| transcribed speech hours:", tot / 3600)

Audio profile-We had to know the sample rate and channels to plan the resampling, how long the segments are for Whisper’s 30 s window, and whether the files match the CSV.	
44.1 kHz mono, 5,857 segments, none over 30 s, CSV durations off by up to 134 s

In [ ]:
import json, glob, os, wave
import numpy as np, pandas as pd

df = pd.read_csv("../data/NewData.csv")

def dur(path):
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate(), w.getframerate(), w.getnchannels()

rows = []
for uid in sorted(df.user_id.unique()):
    p = f"../data/audios/{uid}_audio.wav"
    if not os.path.exists(p):
        print("missing audio:", uid); continue
    d, sr, ch = dur(p)
    cand = df[df.user_id == uid].copy()
    cand["diff"] = (cand.duration - d).abs()
    best = cand.sort_values("diff").iloc[0]
    if len(cand) > 1:
        print("COLLISION", uid, "audio s:", round(d), "csv:", cand.duration.tolist(), "-> picked", best.recording_id)
    rows.append((uid, best.recording_id, d, sr, ch, best["diff"]))

m = pd.DataFrame(rows, columns=["user_id","recording_id","audio_s","sr","ch","dur_diff"])
print(m.sr.value_counts(), m.ch.value_counts())
print("max duration mismatch vs CSV (s):", m.dur_diff.max())

lens, empty, n = [], 0, 0
for f in glob.glob("../data/transcripts/*.json"):
    for s in json.load(open(f, encoding="utf-8")):
        n += 1
        lens.append(s["end"] - s["start"])
        empty += not s["text"].strip()
lens = np.array(lens)
print("segments:", n, "| empty text:", empty)
print("seg length percentiles 5/50/95/max:", np.percentile(lens,[5,50,95]).round(2), lens.max().round(2))
print("segments > 30 s:", (lens > 30).sum())

Transcript vs audio- A transcript running past its audio would mean a truncated file.	
0 truncated files, 20.04 h of audio

In [ ]:
for i, r in m.iterrows():
    segs = json.load(open(f"../data/transcripts/{int(r.user_id)}_transcript.json", encoding="utf-8"))
    m.loc[i, "last_end"] = max(s["end"] for s in segs)
m["over"] = m.last_end - m.audio_s
print("files where transcript ends after audio:", (m.over > 1).sum())
print("audio hours (102 files):", m.audio_s.sum() / 3600)
print(m.sort_values("dur_diff", ascending=False).head(8)[["user_id", "audio_s", "last_end", "dur_diff"]])

40 random samples-Cleaning rules should come from what the labels actually look like, not from guesses.	
Backchannels ('जी', 'हम्म'), Devanagari English, inconsistent punctuation and spelling

In [ ]:
import json, glob, random
random.seed(0)
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += [s["text"] for s in json.load(open(f, encoding="utf-8"))]
for t in random.sample(segs, 40):
    print(repr(t))

Latin, digit and character counts-To find anything that isn’t Devanagari and might break training or WER.	
198 Latin segments, 8 with digits, odd characters _ and -	

In [ ]:
import json, glob, re, collections
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += [s["text"] for s in json.load(open(f, encoding="utf-8"))]

print("with Latin letters:", sum(bool(re.search(r"[A-Za-z]", t)) for t in segs))
print("with digits:", sum(bool(re.search(r"\d", t)) for t in segs))
print("'ब्लैंक' segments:", sum(t.strip() == "ब्लैंक" for t in segs))

chars = collections.Counter(c for t in segs for c in t
                            if not ("\u0900" <= c <= "\u097F" or c == " "))
print(chars.most_common(25))
print([t for t in segs if re.search(r"[A-Za-z\d]", t)][:10])

REDACTED and odd-character inspection-To read the strange cases before deciding what to do with them.	
198 segments are exactly REDACTED, 17 with _, 11 with -, 750 with danda

In [ ]:
import json, glob, re
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += json.load(open(f, encoding="utf-8"))

red = [s for s in segs if "REDACTED" in s["text"]]
print("REDACTED segments:", len(red),
      "| exactly 'REDACTED':", sum(s["text"].strip() == "REDACTED" for s in red),
      "| hours:", round(sum(s["end"] - s["start"] for s in red) / 3600, 3))

def show(label, pat):
    hits = [s["text"] for s in segs if re.search(pat, s["text"]) and "REDACTED" not in s["text"]]
    print(f"\n{label}: {len(hits)}")
    for t in hits[:8]:
        print(repr(t))


show("underscore", r"_")
show("digits", r"\d")
show("dash", r"-")
show("danda", r"[।॥]")
show("devanagari digits", r"[०-९]")

# gpt
show("comma", r",")
show("period", r"\.")
show("exclamation", r"!")
show("colon", r":")
show("danda", r"[।॥]")

chars/sec check - inspects an audio dataset by calculating the speaking rate—measured in characters per second (CPS)—for each audio clip, then prints out the extreme outliers to help identify bad or corrupted data.

In [ ]:
import pandas as pd
m = pd.read_csv("../data_clean/manifest.csv")
m["cps"] = m.text.str.len() / m.duration
pd.set_option("display.max_colwidth", 80)
print(m.sort_values("cps").head(8)[["clip", "duration", "cps", "text"]])
print(m.sort_values("cps").tail(12)[["clip", "duration", "cps", "text"]])
print("clips < 0.3 s:", (m.duration < 0.3).sum())

In [ ]:
import pandas as pd
m = pd.read_csv("../data_clean/manifest.csv")
m["words"] = m.text.str.split().str.len()
m["wps"] = m.words / m.duration
pd.set_option("display.max_colwidth", 70)

print(m.wps.describe(percentiles=[.01, .5, .95, .99]).round(2))
fast  = m[(m.wps > 5) & (m.duration >= 0.5)]
slow  = m[(m.duration > 5) & (m.words <= 2)]
short = m[m.duration < 0.3]
print("fast (>5 words/s, >=0.5 s):", len(fast), "| slow (>5 s, <=2 words):", len(slow), "| <0.3 s:", len(short))
print(fast.sort_values("wps", ascending=False).head(10)[["clip", "duration", "words", "wps", "text"]])
print("######################################")
# print(slow.sort_values("duration", ascending=False).head(10)[["clip", "duration", "words", "text"]])
print(short[["clip", "duration", "text"]].head(10))

In [ ]:
import pandas as pd
m = pd.read_csv("../data_clean/manifest.csv")
m["words"] = m.text.str.split().str.len()
slow = m[(m.duration > 5) & (m.words <= 2)]
print("slow clips:", len(slow), "of", len(m), f"({len(slow)/len(m):.1%})")
print("slow hours:", round(slow.duration.sum()/3600, 2), "of", round(m.duration.sum()/3600, 2))
print("slow clips in val:", (slow.split == "val").sum())

In [ ]:
nbstripout --install
nbstripout --status